# EM - 1D Gaussian Mixture Model (GMM)

Expectation-Maximization (EM) finds maximum likelihood estimates when data have latent component assignments. We alternate between estimating responsibilities (E-step) and updating parameters (M-step).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def normal_pdf(x, mu, sigma):
    return (1/(np.sqrt(2*np.pi)*sigma)) * np.exp(-0.5*((x-mu)/sigma)**2)

In [ ]:
# Generate mixture data
np.random.seed(1)
n = 200
# true params
mus_true = np.array([0.0, 5.0])
sigmas_true = np.array([0.8, 1.0])
pis_true = np.array([0.4, 0.6])

# sample
components = np.random.choice(2, size=n, p=pis_true)
x = np.array([np.random.normal(mus_true[k], sigmas_true[k]) for k in components])
plt.hist(x, bins=30, density=True, alpha=0.6)
plt.title('Histogram of mixture data')
plt.show()

In [ ]:
# Initialize params
np.random.seed(2)
K = 2
mus = np.random.choice(x, K, replace=False).astype(float)
sigmas = np.ones(K)*1.0
pis = np.ones(K)/K

def log_likelihood(x, mus, sigmas, pis):
    comp = np.array([pis[k]*normal_pdf(x, mus[k], sigmas[k]) for k in range(K)])
    return np.sum(np.log(comp.sum(axis=0)))

lls = []
for it in range(20):
    # E-step: responsibilities r[n,k]
    resp = np.zeros((n,K))
    for k in range(K):
        resp[:,k] = pis[k]*normal_pdf(x, mus[k], sigmas[k])
    resp = resp / resp.sum(axis=1, keepdims=True)
    # M-step:
    N_k = resp.sum(axis=0)
    for k in range(K):
        mus[k] = (resp[:,k]*x).sum() / N_k[k]
        sigmas[k] = np.sqrt((resp[:,k]* (x - mus[k])**2).sum() / N_k[k])
        pis[k] = N_k[k] / n
    ll = log_likelihood(x, mus, sigmas, pis)
    lls.append(ll)
    if it % 5 == 0:
        print(f'iter {it:2d} ll={ll:.3f}')

print('Final params:')
print('mus', mus)
print('sigmas', sigmas)
print('pis', pis)

In [ ]:
# Plot histogram and learned components
xs = np.linspace(x.min()-1, x.max()+1, 400)
plt.hist(x, bins=30, density=True, alpha=0.5)
for k in range(K):
    plt.plot(xs, pis[k]*normal_pdf(xs, mus[k], sigmas[k]), label=f'comp {k}')
plt.legend()
plt.title('Learned mixture components')
plt.show()

# Plot log-likelihood over iterations
plt.plot(lls, marker='o')
plt.xlabel('Iteration')
plt.ylabel('Log-likelihood')
plt.title('EM log-likelihood')
plt.show()

**Interpretation:** The EM algorithm increases log-likelihood each iteration and the learned component means should be close to the true means (0 and 5) up to label swapping.